# Sliding Window — Part 1: Introduction

This notebook covers the **fundamentals** of the sliding window technique.

---

## What is a Window?

A **window** is just a contiguous slice of an array or string defined by two pointers:
- `l` — left boundary
- `r` — right boundary

```
arr = [1, 3, 2, 5, 4, 7]
          l        r
          ^--------^
          window = [2, 5, 4]
```

Window size = `r - l + 1`

## Why Not Brute Force?

Brute force checks **every possible subarray** — O(n²) or worse.

```python
for i in range(n):
    for j in range(i, n):
        # check subarray arr[i..j]
```

**Problem:** When you move from window `[i..j]` to `[i..j+1]`, you recompute everything from scratch.

**Sliding window insight:** When you move the right pointer one step, you only need to ADD one element. When you move the left pointer, you only need to REMOVE one element. No recomputation.

## Two Flavors of Sliding Window

### 1. Fixed-Size Window
Window size `k` is given. Slide it across the array.

```
[1, 3, -1, -3, 5, 3]   k=3
 ^--^                   window 1
    ^--^                window 2
       ^--^             window 3 ...
```

### 2. Variable-Size Window
Window expands/shrinks based on a condition. This is the more common and tricky type.

---
## Fixed-Size Window: Maximum Sum Subarray of Size K

In [ ]:
def max_sum_fixed(arr, k):
    """
    Find the maximum sum of any subarray of size k.
    
    Strategy: build first window of size k, then slide —
    add incoming element on right, remove outgoing element on left.
    """
    n = len(arr)
    if n < k:
        return -1

    # Build first window
    window_sum = sum(arr[:k])
    max_sum    = window_sum

    for i in range(k, n):
        window_sum += arr[i]         # add new right element
        window_sum -= arr[i - k]     # remove old left element
        max_sum = max(max_sum, window_sum)

    return max_sum


print(max_sum_fixed([2, 1, 5, 1, 3, 2], 3))   # 9  (5+1+3)
print(max_sum_fixed([2, 3, 4, 1, 5],    2))   # 7  (3+4)

### Visualising the slide:
```
arr = [2, 1, 5, 1, 3, 2]   k=3

i=0: [2, 1, 5]  sum=8
i=1: [1, 5, 1]  sum=7   (+1, -2)
i=2: [5, 1, 3]  sum=9   (+3, -1)  ← max
i=3: [1, 3, 2]  sum=6   (+2, -5)
```

---
## Variable-Size Window: The Template

```python
l = 0

for r in range(len(arr)):
    # 1. EXPAND: add arr[r] to your window state

    # 2. SHRINK: while window is INVALID, remove arr[l] and move l forward
    while <window is invalid>:
        # remove arr[l] from state
        l += 1

    # 3. RECORD: window [l..r] is now valid — update your answer
```

The key question is always: **what makes a window invalid?**

---
## Example 1: Longest Subarray with Sum ≤ Target

In [ ]:
def longest_subarray_sum_leq(arr, target):
    """
    Find the longest subarray whose sum is <= target.
    Window invalid when: window_sum > target
    """
    l          = 0
    window_sum = 0
    max_len    = 0

    for r in range(len(arr)):
        window_sum += arr[r]              # expand

        while window_sum > target:        # shrink until valid
            window_sum -= arr[l]
            l += 1

        max_len = max(max_len, r - l + 1) # record

    return max_len


print(longest_subarray_sum_leq([1, 2, 3, 4, 5], 9))    # 4  ([1,2,3] or [2,3,4]...)
print(longest_subarray_sum_leq([3, 1, 2, 7, 4, 2], 8)) # 4  ([3,1,2] or [1,2,7]...)

---
## Example 2: Longest Substring with At Most K Distinct Characters

In [ ]:
from collections import defaultdict

def longest_k_distinct(s, k):
    """
    Longest substring with at most k distinct characters.
    Window invalid when: len(freq_map) > k
    """
    freq = defaultdict(int)
    l    = 0
    best = 0

    for r in range(len(s)):
        freq[s[r]] += 1                   # expand: add right char

        while len(freq) > k:              # shrink: too many distinct chars
            freq[s[l]] -= 1
            if freq[s[l]] == 0:
                del freq[s[l]]
            l += 1

        best = max(best, r - l + 1)       # record

    return best


print(longest_k_distinct("araaci", 2))   # 4  ("araa")
print(longest_k_distinct("araaci", 1))   # 2  ("aa")
print(longest_k_distinct("cbbebi", 3))   # 5  ("cbbeb")

---
## The Shrink Condition is Everything

| Problem | Invalid condition | What to track |
|---------|-------------------|---------------|
| Sum ≤ target | `sum > target` | running sum |
| At most K distinct chars | `len(map) > k` | frequency map |
| No duplicate chars | `char already in window` | seen set |
| At most K zeros | `zero_count > k` | zero counter |
| Char replacement | `len - maxFreq > k` | frequency + max |

Once you identify what makes a window **invalid**, the template is always the same.

---
## Common Mistakes

### Mistake 1: Moving l backward
```python
# WRONG — l should never go backward
if duplicate found:
    l = seen[char]   # this can move l left if char is outside window!

# CORRECT
l = max(l, seen[char] + 1)   # only move forward
```

### Mistake 2: Checking validity AFTER recording
```python
# WRONG order
for r in range(n):
    # expand
    max_len = max(max_len, r - l + 1)   # recorded BEFORE shrinking!
    while invalid: l += 1

# CORRECT order
for r in range(n):
    # expand
    while invalid: l += 1               # shrink first
    max_len = max(max_len, r - l + 1)   # then record
```

---
## Summary

```
FIXED window  → slide l and r together, window size stays constant
VARIABLE window → r always moves forward, l moves forward only when window goes invalid

Template:
  expand r
  while invalid: shrink l
  record answer
```

Next: **Part 2** covers the two types of variable-window problems and why counting problems need a different strategy.